In [ ]:
!pip install matplotlib scipy

In [83]:
import json
import pandas as pd
import json
import numpy as np
import matplotlib.pyplot as plt
from glob import glob
from scipy.stats import t

In [ ]:
# Read constraints data
constraints_df = pd.read_csv('constraints_attribution.csv').reset_index(drop=True)
constraints_ldp_df = constraints_df[constraints_df['chain'] == True]
constraints_ldp_df['constraints'] = constraints_ldp_df['non-linear constraints'] + constraints_ldp_df['linear constraints']
constraints_ldp_df = constraints_ldp_df[['k', 'constraints']]


constraints_no_ldp_df = constraints_df[constraints_df['chain'] == False]
constraints_no_ldp_df['constraints'] = constraints_no_ldp_df['non-linear constraints'] + constraints_no_ldp_df['linear constraints']
constraints_no_ldp_df = constraints_no_ldp_df[['k', 'constraints']]

constraints_merged = pd.merge(constraints_no_ldp_df, constraints_ldp_df, on='k', suffixes=('_no_chain', '_chain'), how='outer')
constraints_merged = constraints_merged.rename(columns={'constraints_no_chain': 'no_chain_constraints', 'constraints_chain': 'chain_constraints'})


In [86]:
constraints_merged['k'] = constraints_merged['k'].astype(int)

In [ ]:
# Generate LaTeX table
latex_table = "\\begin{table*}[htbp]\n"
latex_table += "\\centering\n"
latex_table += "\\caption{Number of constraints for different credential history sizes and the streaming algorithm implementation}\n"
latex_table += "\\label{tab:constraints}\n"
latex_table += "\\begin{tabular}{@{}lcc@{}}\n"
latex_table += "\\toprule\n"
latex_table += "Credential History Size & Constraints (Without RAPPOR) & Constraints (With RAPPOR) \\\\\n"
latex_table += "\\midrule\n"

for _, row in constraints_merged.sort_values('k').iterrows():
    k = row['k']
    if k == 'streaming':
        k_str = 'streaming'
    else:
        k_str = str(int(k))
    latex_table += f"{k_str} & "
    latex_table += f"{int(row['no_ldp_constraints']):,} & "
    latex_table += f"{int(row['ldp_constraints']):,} \\\\\n"
    # latex_table += "\\hline\n"

latex_table += "\\bottomrule\n"
latex_table += "\\end{tabular}\n"
latex_table += "\\end{table}"

print(latex_table)

In [ ]:
def load_results(path):
    with open(path, 'r') as f:
        data = json.load(f)
    data = [d for d in data if d.get('proveError') is None and d.get('verifyError') is None]
    prove = np.array([d['proveTimeMs'] for d in data])
    verify = np.array([d['verifyTimeMs'] for d in data])
    return prove, verify

def mean_ci(arr):
    n = len(arr)
    mean = np.mean(arr)
    stderr = np.std(arr, ddof=1) / np.sqrt(n)
    ci = t.ppf(0.975, n-1) * stderr
    return mean, ci

def get_k_from_filename(fname):
    return int(fname.split('_k')[-1].split('.')[0])

def aggregate_results(result_dir):
    files = sorted(glob(f"{result_dir}/experiment_results_k*.json"), key=get_k_from_filename)
    ks, means, cis, means_verify, cis_verify = [], [], [], [], []
    for f in files:
        k = get_k_from_filename(f)
        prove, verify = load_results(f)
        m, ci = mean_ci(prove)
        m_v, ci_v = mean_ci(verify)
        ks.append(k)
        means.append(m)
        cis.append(ci)
        means_verify.append(m_v)
        cis_verify.append(ci_v)
    return np.array(ks), np.array(means), np.array(cis), np.array(means_verify), np.array(cis_verify)

def load_streaming(result_dir):
    prove, verify = load_results(f"{result_dir}/experiment_results_streaming.json")
    m, ci = mean_ci(prove)
    m_v, ci_v = mean_ci(verify)
    return m, ci, m_v, ci_v

def plot_broken_axis(ldp_dir, noldp_dir, streaming_ldp_dir, streaming_noldp_dir, title_prefix=""):
    ks, means, cis, means_verify, cis_verify = aggregate_results(ldp_dir)
    ks_noldp, means_noldp, cis_noldp, means_verify_noldp, cis_verify_noldp = aggregate_results(noldp_dir)
    # Streaming
    m_stream_ldp, ci_stream_ldp, m_v_stream_ldp, ci_v_stream_ldp = load_streaming(streaming_ldp_dir)
    m_stream_noldp, ci_stream_noldp, m_v_stream_noldp, ci_v_stream_noldp = load_streaming(streaming_noldp_dir)

    # Proving time
    fig, (ax_main, ax_stream) = plt.subplots(1, 2, gridspec_kw={'width_ratios': [4, 1]}, sharey=True, figsize=(10,5))
    fig.suptitle(f'{title_prefix}Proving Time for Varying Credential History Size and Streaming')
    # Main plot
    ax_main.errorbar(ks, means, yerr=cis, fmt='-o', label='With RAPPOR')
    ax_main.errorbar(ks_noldp, means_noldp, yerr=cis_noldp, fmt='-o', label='Without RAPPOR')
    ax_main.set_xscale('log')
    ax_main.set_yscale('log')
    ax_main.set_xlabel('Credential history size')
    ax_main.set_ylabel('Proving time (ms)')
    # Set y-axis limits to start slightly below minimum value
    y_min = min(np.min(means - cis), np.min(means_noldp - cis_noldp))
    ax_main.set_ylim(bottom=y_min * 0.8)
    # ax_main.set_title(f'{title_prefix}Proving Time')
    ax_main.legend()
    # Remove right spine
    ax_main.spines['right'].set_visible(False)
    ax_main.yaxis.tick_left()
    ax_main.tick_params(labelright=False)
    # Streaming plot
    ax_stream.errorbar([1], [m_stream_ldp], yerr=[ci_stream_ldp], fmt='s', color='red', label='With RAPPOR', markersize=10, capsize=5)
    ax_stream.errorbar([2], [m_stream_noldp], yerr=[ci_stream_noldp], fmt='D', color='blue', label='Without RAPPOR', markersize=10, capsize=5)
    ax_stream.set_xticks([])
    ax_stream.set_xlim(0.5,2.5)
    ax_stream.set_xlabel('Streaming')
    ax_stream.legend(loc='upper right')
    ax_stream.spines['left'].set_visible(False)
    ax_stream.yaxis.tick_right()
    # Diagonal lines to indicate break
    d = .01
    kwargs = dict(transform=ax_main.transAxes, color='k', clip_on=False)
    ax_main.plot([1-d,1+d], [-d,+d], **kwargs)
    ax_main.plot([1-d,1+d],[1-d,1+d], **kwargs)
    kwargs.update(transform=ax_stream.transAxes)
    ax_stream.plot([-d,+d], [1-d,1+d], **kwargs)
    ax_stream.plot([-d,+d], [-d,+d], **kwargs)
    plt.tight_layout()
    plt.savefig("graphics/proving.pdf")
    plt.show()

    # Verifying time (linear y)
    fig, (ax_main, ax_stream) = plt.subplots(1, 2, gridspec_kw={'width_ratios': [4, 1]}, sharey=True, figsize=(10,5))
    fig.suptitle(f'{title_prefix}Verifying Time for Varying Credential History Size and Streaming')
    ax_main.errorbar(ks, means_verify, yerr=cis_verify, fmt='-o', label='With RAPPOR')
    ax_main.errorbar(ks_noldp, means_verify_noldp, yerr=cis_verify_noldp, fmt='-o', label='Without RAPPOR')
    ax_main.set_xscale('log')
    ax_main.set_xlabel('Credential history size')
    ax_main.set_ylabel('Verifying time (ms)')
    ax_main.legend(loc='upper left')
    ax_main.spines['right'].set_visible(False)
    ax_main.yaxis.tick_left()
    ax_main.tick_params(labelright=False)
    ax_stream.errorbar([1], [m_v_stream_ldp], yerr=[ci_v_stream_ldp], fmt='s', color='red', label='With RAPPOR', markersize=10, capsize=5)
    ax_stream.errorbar([2], [m_v_stream_noldp], yerr=[ci_v_stream_noldp], fmt='D', color='blue', label='Without RAPPOR', markersize=10, capsize=5)
    ax_stream.set_xlabel('Streaming')
    ax_stream.set_xlim(0.5,2.5)
    ax_stream.set_xticks([])  # Remove x ticks
    ax_stream.legend(loc='upper right')
    ax_stream.spines['left'].set_visible(False)
    ax_stream.yaxis.tick_right()
    d = .01
    kwargs = dict(transform=ax_main.transAxes, color='k', clip_on=False)
    ax_main.plot([1-d,1+d], [-d,+d], **kwargs)
    ax_main.plot([1-d,1+d],[1-d,1+d], **kwargs)
    kwargs.update(transform=ax_stream.transAxes)
    ax_stream.plot([-d,+d], [1-d,1+d], **kwargs)
    ax_stream.plot([-d,+d], [-d,+d], **kwargs)
    plt.tight_layout()
    plt.savefig("graphics/verifying.pdf")
    plt.show()

# --- Usage ---
plot_broken_axis(
    ldp_dir="results_ldp",
    noldp_dir="results_no_ldp",
    streaming_ldp_dir="results_ldp",
    streaming_noldp_dir="results_no_ldp"
)

In [ ]:
# --- Attribution Results Plotting ---

def plot_attribution_results(chain_dir, single_dir, title_prefix=""):
    ks_chain, means_chain, cis_chain, means_verify_chain, cis_verify_chain = aggregate_results(chain_dir)
    ks_single, means_single, cis_single, means_verify_single, cis_verify_single = aggregate_results(single_dir)

    # Proving time
    plt.figure(figsize=(8,5))
    plt.errorbar(ks_chain, means_chain, yerr=cis_chain, fmt='-o', label='Chain Attribution')
    plt.errorbar(ks_single, means_single, yerr=cis_single, fmt='-o', label='Single Attribution')
    plt.xscale('log')
    plt.yscale('log')
    plt.xlabel('Attribution history size')
    plt.ylabel('Proving time (ms)')
    plt.title(f'{title_prefix}Proving Time for Attribution by Attribution History Size')
    plt.legend(loc='upper left')
    plt.tight_layout()
    plt.savefig("graphics/attribution_proving.pdf")
    plt.show()

    # Verifying time
    plt.figure(figsize=(8,5))
    plt.errorbar(ks_chain, means_verify_chain, yerr=cis_verify_chain, fmt='-o', label='Chain Attribution')
    plt.errorbar(ks_single, means_verify_single, yerr=cis_verify_single, fmt='-o', label='Single Attribution')
    plt.xscale('log')
    plt.xlabel('Attribution history size')
    plt.ylabel('Verifying time (ms)')
    plt.title(f'{title_prefix}Verifying Time for Attribution by Attribution History Size')
    plt.legend(loc='upper left')
    plt.tight_layout()
    plt.savefig("graphics/attribution_verifying.pdf")
    plt.show()

# --- Usage ---
plot_attribution_results(
    chain_dir="results_attribution_chain",
    single_dir="results_attribution_single"
)